In [2]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from Config import Config
from datetime import datetime
from Image_Imbedding import ImageEmbedding
from Image_Transformer import ImgTransformer
from Image_Util import show_img_tensor_CHW

from Fliker_Comment_Tokenizer import FlikerCommentTokenizer
from Fliker_Image_Comment_Dataset import ImgCommentDataset
from Model_Util import count_parameters
from pathlib import Path
from text_token_embedding import TextTokenEmbedding
from text_casual_mask_transformer import TextMaskedTransformer
from vlm_model import ImgLanguageModel
from vlm_train import train
from torch.utils.tensorboard import SummaryWriter


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.profiler import profile, record_function, ProfilerActivity
from torch.utils.data import DataLoader
import torchvision.transforms.functional as VF

plt.rcParams["savefig.bbox"] = 'tight'

In [ ]:
train()

In [ ]:
model_trained = ImgLanguageModel(config=config)
model_trained.load_state_dict(torch.load(model_path))
model_trained = model_trained.to(device)
model_trained.eval()

In [ ]:
batch_img_tensor, batch_img_id_tensor, batch_target_tensor, batch_target_mask_tensor = (
    next(iter(test_dataloader))
)
batch_img_tensor = batch_img_tensor.to(device)
batch_target_tensor = batch_target_tensor.to(device)
batch_target_mask_tensor = batch_target_mask_tensor.to(device)
img_loss, text_loss, img_contrastive_prob, text_contrastive_prob, lm_loss, lm_logit = (
    model_trained(batch_img_tensor, batch_target_tensor)
)

In [ ]:
print(img_loss)
show_img_tensor_CHW(batch_img_tensor[19].cpu())

In [ ]:
torch.argmax(img_contrastive_prob, dim=0)

In [ ]:
model_trained.text_transformer.text_token_embedding.text_encoder.decode(
    batch_target_tensor[19]
)

In [ ]:
def show(imgs, comments, labels):
    if not isinstance(imgs, list):
        imgs = [imgs]
    imgs_per_row = 1
    fix, axs = plt.subplots(
        nrows=(len(imgs) + imgs_per_row - 1) // imgs_per_row,
        ncols=imgs_per_row,
        squeeze=False,
        figsize=(16, 60),
    )
    for i, img in enumerate(imgs):
        img = img.detach()
        img = VF.to_pil_image(img)
        row = i // imgs_per_row
        col = i % imgs_per_row
        axs[row, col].imshow(np.asarray(img))
        axs[row, col].set(xticklabels=[], yticklabels=[], xticks=[], yticks=[])
        title = f'pred: {comments[i].replace("<pad>", "").replace("<bos>", "")}\nlabel: {labels[i].replace("<pad>", "").replace("<bos>", "")}'
        axs[row, col].set_title(title)

In [ ]:
img_predicted_commments_index = torch.argmax(img_contrastive_prob, dim=0)

show(
    imgs=[img for img in batch_img_tensor.cpu()],
    labels=[
        model_trained.text_transformer.text_token_embedding.text_encoder.decode(
            target_tensor
        )
        for target_tensor in batch_target_tensor
    ],
    comments=[
        model_trained.text_transformer.text_token_embedding.text_encoder.decode(
            batch_target_tensor[predicted_comment_index]
        )
        for predicted_comment_index in img_predicted_commments_index
    ],
)